# Hamilton 32-position bare tube-carrier geometry test

This notebook tests `hamilton_tube_carrier_32_a00` **without Eppendorf inserts and without tubes**. The bare carrier is loaded at rail 19. A 1000 µL filtered-tip rack is placed in tip-carrier position 0 at rail 25.

Physical channel 1 (PyLabRobot channel 0) visits the modeled centers of the frontmost and rearmost carrier sites. At each site it stops 5 mm above the modeled 60.0 mm carrier-top plane, then descends to the carrier-top height and pauses for operator inspection.

**Remove every insert and tube, keep the deck clear, supervise every move, and execute the cells from top to bottom once.** Do not rerun the deck-layout cell after setup because it creates a new, uninitialized `LiquidHandler`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources import hamilton_96_tiprack_1000uL_filter
from pylabrobot.resources.hamilton import (
    STARLetDeck,
    TIP_CAR_480_A00,
    hamilton_tube_carrier_32_a00,
)


In [ ]:
# PyLabRobot channels, carrier positions, and carrier sites are zero-based.
CHANNEL = 0  # physical channel 1
TIP_CARRIER_POSITION = 0
TIP_POSITION = "A1"
TUBE_CARRIER_RAIL = 19
TIP_CARRIER_RAIL = 25
TEST_SITES = (0, 31)
SAFE_TRAVERSE_Z = 247.6
ABOVE_TOP_CLEARANCE_MM = 5.0

# Edit these after physical inspection. Positive values follow deck coordinates.
X_CALIBRATION_OFFSET_MM = 0.0
Y_CALIBRATION_OFFSET_MM = 0.0
Z_CALIBRATION_OFFSET_MM = 0.0

backend = STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())

tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
tip_rack_1000ul = hamilton_96_tiprack_1000uL_filter(
    name="tips_1000ul_filter"
)
tip_carrier[TIP_CARRIER_POSITION].assign_child_resource(tip_rack_1000ul)
lh.deck.assign_child_resource(tip_carrier, rails=TIP_CARRIER_RAIL)

tube_carrier = hamilton_tube_carrier_32_a00(name="tube_carrier")
lh.deck.assign_child_resource(tube_carrier, rails=TUBE_CARRIER_RAIL)


In [ ]:
# Validate and report the resource parameters before connecting to the robot.
site_centers = [
    tube_carrier[index].get_absolute_location(x="c", y="c")
    for index in range(32)
]
carrier_top_z = tube_carrier.get_absolute_location(z="t").z
modeled_pitch_y = site_centers[1].y - site_centers[0].y
center_x_values = [center.x for center in site_centers]

assert tube_carrier.get_size_x() == 22.5
assert tube_carrier.get_size_y() == 497.0
assert tube_carrier.get_size_z() == 60.0
assert len(tube_carrier.sites) == 32
assert tube_carrier[0].get_size_x() == 13.4
assert tube_carrier[0].get_size_y() == 13.4
assert tube_carrier[0].get_location_wrt(tube_carrier).z == 50.2
assert modeled_pitch_y == 15.0
assert all(
    abs(center.x - center_x_values[0]) < 1e-9 for center in site_centers
), "All 32 sites must share one X centerline"
assert site_centers[-1].y > site_centers[0].y
assert tip_rack_1000ul[TIP_POSITION][0].has_tip()
selected_tip = tip_rack_1000ul[TIP_POSITION][0].get_tip()
assert selected_tip.nominal_volume == 1000
assert selected_tip.has_filter
assert carrier_top_z + ABOVE_TOP_CLEARANCE_MM < SAFE_TRAVERSE_Z

print(lh.deck.summary())
print("\n=== BARE CARRIER MODEL (mm) ===")
print(f"Dimensions:       {tube_carrier.get_size_x():.1f} x "
      f"{tube_carrier.get_size_y():.1f} x {tube_carrier.get_size_z():.1f}")
print(f"Site count:       {len(tube_carrier.sites)}")
print(f"Bare-hole diameter: {tube_carrier[0].get_size_x():.2f}")
print(f"Tube-support Z:     {tube_carrier[0].get_location_wrt(tube_carrier).z:.2f}")
print(f"Site pitch Y:     {modeled_pitch_y:.2f}")
print(f"Site centerline X:{center_x_values[0]:.2f}")
print(f"Front site center:X={site_centers[0].x:.2f}, Y={site_centers[0].y:.2f}")
print(f"Rear site center: X={site_centers[-1].x:.2f}, Y={site_centers[-1].y:.2f}")
print(f"Carrier top Z:    {carrier_top_z:.2f}")
print(f"Approach Z:       {carrier_top_z + ABOVE_TOP_CLEARANCE_MM:.2f}")
print(f"Safe traverse Z:  {SAFE_TRAVERSE_Z:.2f}")
print("Bare-hole diameter and tube-support Z are directly measured values.")


## Connect and run

Confirm the printed deck layout matches the physical deck. Rail 19 must contain the completely bare 32-position carrier. Rail 25 must contain the tip carrier with a 1000 µL filtered-tip rack in position 0 and a tip at A1.

In [ ]:
if lh.setup_finished:
    print("Liquid handler setup is already complete; setup was not repeated.")
else:
    await lh.setup(skip_autoload=True)


In [ ]:
def confirm(message: str):
    response = input(f"\n{message} Type GO and press Enter to continue: ").strip().upper()
    if response != "GO":
        raise RuntimeError("Operator cancelled the geometry test.")


async def run_bare_carrier_geometry_test(
    x_offset_mm: float = X_CALIBRATION_OFFSET_MM,
    y_offset_mm: float = Y_CALIBRATION_OFFSET_MM,
    z_offset_mm: float = Z_CALIBRATION_OFFSET_MM,
):
    top_z = carrier_top_z + z_offset_mm
    approach_z = top_z + ABOVE_TOP_CLEARANCE_MM
    assert approach_z < SAFE_TRAVERSE_Z
    tip_is_attached = False

    print("\n=== CALIBRATED TARGETS (mm) ===")
    print(f"Offsets: X={x_offset_mm:+.2f}, Y={y_offset_mm:+.2f}, Z={z_offset_mm:+.2f}")
    print(f"Top plane Z={top_z:.2f}; approach Z={approach_z:.2f}")

    confirm(
        "Verify the bare carrier is at rail 19; ALL INSERTS AND TUBES are "
        "removed; the 1000 µL filtered tip is at position 0, A1; deck clear."
    )
    await lh.pick_up_tips(tip_rack_1000ul[TIP_POSITION], use_channels=[CHANNEL])
    tip_is_attached = True

    try:
        await lh.prepare_for_manual_channel_operation(CHANNEL)
        await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)

        for site_index in TEST_SITES:
            center = site_centers[site_index]
            target_x = center.x + x_offset_mm
            target_y = center.y + y_offset_mm

            confirm(
                f"Ready to traverse to site {site_index} center at "
                f"X={target_x:.2f}, Y={target_y:.2f}."
            )
            await lh.move_channel_x(CHANNEL, target_x)
            await lh.move_channel_y(CHANNEL, target_y)

            confirm(
                f"Ready to lower to Z={approach_z:.2f}, "
                f"{ABOVE_TOP_CLEARANCE_MM:.1f} mm above the calibrated top plane."
            )
            await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=approach_z)
            print(f"Inspect X/Y alignment over bare site {site_index}.")

            confirm(
                f"Ready to descend to the modeled carrier-top height, "
                f"Z={top_z:.2f}, at site {site_index}."
            )
            await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=top_z)
            print(f"Tip is at the carrier-top height for site {site_index}.")
            confirm(
                "Pause for inspection. When inspection is complete, confirm "
                "to raise the tip to safe traverse Z."
            )

            await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)

        confirm("Both sites inspected; ready to return the tip to A1.")
        await lh.drop_tips(tip_rack_1000ul[TIP_POSITION], use_channels=[CHANNEL])
        tip_is_attached = False
        print("Bare-carrier geometry test complete; tip returned to A1.")
    finally:
        if tip_is_attached:
            print("Test interrupted with a tip attached; attempting a safe raise.")
            try:
                await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
                print("Channel raised. Run the recovery cell to return the tip.")
            except Exception as recovery_error:
                print(f"Automatic raise failed: {recovery_error!r}")
                print("Recover the channel manually before moving deck hardware.")


In [ ]:
await run_bare_carrier_geometry_test(
    x_offset_mm=X_CALIBRATION_OFFSET_MM,
    y_offset_mm=Y_CALIBRATION_OFFSET_MM,
    z_offset_mm=Z_CALIBRATION_OFFSET_MM,
)


## Recovery

Run this only if the test was interrupted with the 1000 µL tip still attached.

In [ ]:
await lh.backend.move_channel_tool_z(channel_idx=CHANNEL, z=SAFE_TRAVERSE_Z)
await lh.drop_tips(tip_rack_1000ul[TIP_POSITION], use_channels=[CHANNEL])


## Stop

Run after the test or recovery is complete.

In [ ]:
await lh.stop()
